# 03 - Model Training (baseline)
Trains a Logistic Regression baseline on the **saved** train split and saves the model.
We never re-split the data here, so the test set stays untouched.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # so we can import from src/

import joblib
from src import config
from src.data_preprocessing import load_processed_data

X_train, X_test, y_train, y_test = load_processed_data()
print(X_train.shape, X_test.shape)
print("Default rate  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

### Handle class imbalance (train data only)
Only ~22% of customers default. SMOTE creates synthetic default examples in the **training** set only.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=config.RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", y_train_sm.value_counts().to_dict())

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_sm, y_train_sm)
print(model)

### Evaluate on the untouched test set

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_prob), 4))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline Logistic Regression")
plt.show()

### Save the model (one file only)

In [ ]:
config.MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(model, config.MODEL_PATH)
print("Model saved at", config.MODEL_PATH)